# Advanced Model Comparison

Fair multi-model comparison with proper tuning, dedicated boosting libraries, and a
text-based classifier on raw agent responses.

**Models:**
- Logistic Regression (tuned baseline)
- Random Forest (tuned)
- XGBoost (tuned)
- LightGBM (tuned)
- SVM RBF (tuned)
- MLP (tuned)
- TF-IDF + Logistic Regression on raw response text

**Protocol:**
- Stratified 5-fold CV (outer) for evaluation
- RandomizedSearchCV (inner 3-fold) for hyperparameter tuning
- Class-weight balancing throughout
- All results logged to `results/model_comparison.json`

In [ ]:
import warnings
warnings.filterwarnings('ignore')

import json
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from datetime import datetime

from sklearn.model_selection import (
    StratifiedKFold, RandomizedSearchCV, cross_validate
)
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from sklearn.neural_network import MLPClassifier
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics import (
    roc_auc_score, average_precision_score, f1_score,
    matthews_corrcoef, make_scorer
)
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier

sns.set_theme(style='whitegrid', palette='colorblind', font_scale=1.1)
plt.rcParams['figure.dpi'] = 120

SEED = 42
np.random.seed(SEED)
FIGURES_DIR = Path('../reports/figures')
RESULTS_DIR = Path('../results')
FIGURES_DIR.mkdir(parents=True, exist_ok=True)
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

print('Libraries loaded.')

## 1. Load Data (Structured Features + Raw Text)

In [ ]:
# Load structured features and labels
d2c = pd.read_parquet('../data/processed/d2c_labels.parquet')
d2d = pd.read_parquet('../data/processed/d2d_features.parquet')
d2b = pd.read_parquet('../data/processed/d2b_turns.parquet')

# Merge features with labels
df = d2d.merge(d2c[['session_id', 'overall_unsafe']], on='session_id')

# Target — force to plain numpy
y = np.array(df['overall_unsafe'].astype(int).tolist())

# Numeric features
numeric_cols = [
    'response_length_words', 'response_length_chars', 'sentence_count',
    'avg_sentence_length', 'paragraph_count', 'hedging_count', 'hedging_freq',
    'certainty_count', 'certainty_freq', 'hedging_certainty_ratio',
    'nct_id_count', 'nct_id_unique', 'citation_density',
    'bullet_count', 'numbered_list_count', 'refusal_count',
    'is_monitored', 'is_baseline_pressure'
]
bool_cols = ['has_refusal', 'has_citations', 'has_hedging', 'has_certainty']
for c in bool_cols:
    df[c] = df[c].astype(int)

cat_cols = ['scenario_id', 'pressure_id', 'monitoring_id']
le_dict = {}
for c in cat_cols:
    le = LabelEncoder()
    df[f'{c}_enc'] = le.fit_transform(df[c].astype(str))
    le_dict[c] = le
encoded_cat_cols = [f'{c}_enc' for c in cat_cols]

feature_cols = numeric_cols + bool_cols + encoded_cat_cols
X_struct = np.array(df[feature_cols].values.tolist(), dtype=float)

# Raw text: concatenate all assistant turns per session
text_df = d2b[d2b['role'] == 'assistant'].groupby('session_id')['content'].apply(
    lambda x: ' '.join(x)).reset_index()
text_df.columns = ['session_id', 'full_response']

# Align text with structured data
df = df.merge(text_df, on='session_id', how='left')
df['full_response'] = df['full_response'].fillna('')
X_text = df['full_response'].tolist()  # plain Python list

print(f'Structured features: {X_struct.shape}')
print(f'Text responses: {len(X_text)} (avg {np.mean([len(t) for t in X_text]):.0f} chars)')
print(f'Target: {np.bincount(y)} (0=safe, 1=unsafe)')

## 2. Define Models with Hyperparameter Search Spaces

In [ ]:
# Outer CV for evaluation
outer_cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)
# Inner CV for tuning
inner_cv = StratifiedKFold(n_splits=3, shuffle=True, random_state=SEED)

# Imbalance ratio for scale_pos_weight
imbalance_ratio = (y == 0).sum() / (y == 1).sum()

# Model definitions with search spaces
model_configs = {
    'Logistic Regression': {
        'pipeline': Pipeline([
            ('scaler', StandardScaler()),
            ('clf', LogisticRegression(class_weight='balanced', max_iter=2000, random_state=SEED))
        ]),
        'params': {
            'clf__C': [0.01, 0.1, 0.5, 1.0, 5.0, 10.0],
            'clf__penalty': ['l1', 'l2'],
            'clf__solver': ['saga'],
        },
        'n_iter': 12,
    },
    'Random Forest': {
        'pipeline': Pipeline([
            ('scaler', StandardScaler()),
            ('clf', RandomForestClassifier(class_weight='balanced', random_state=SEED))
        ]),
        'params': {
            'clf__n_estimators': [100, 200, 300, 500],
            'clf__max_depth': [3, 5, 7, 10, None],
            'clf__min_samples_leaf': [3, 5, 10, 15],
            'clf__max_features': ['sqrt', 'log2', 0.5],
        },
        'n_iter': 30,
    },
    'XGBoost': {
        'pipeline': Pipeline([
            ('scaler', StandardScaler()),
            ('clf', XGBClassifier(
                scale_pos_weight=imbalance_ratio, eval_metric='logloss',
                use_label_encoder=False, random_state=SEED, verbosity=0))
        ]),
        'params': {
            'clf__n_estimators': [100, 200, 300, 500],
            'clf__max_depth': [3, 5, 7],
            'clf__learning_rate': [0.01, 0.05, 0.1, 0.2],
            'clf__subsample': [0.7, 0.8, 0.9, 1.0],
            'clf__colsample_bytree': [0.6, 0.8, 1.0],
            'clf__reg_alpha': [0, 0.1, 1.0],
            'clf__reg_lambda': [1.0, 2.0, 5.0],
        },
        'n_iter': 40,
    },
    'LightGBM': {
        'pipeline': Pipeline([
            ('scaler', StandardScaler()),
            ('clf', LGBMClassifier(
                is_unbalance=True, random_state=SEED, verbosity=-1, n_jobs=1))
        ]),
        'params': {
            'clf__n_estimators': [100, 200, 300, 500],
            'clf__max_depth': [3, 5, 7, -1],
            'clf__learning_rate': [0.01, 0.05, 0.1, 0.2],
            'clf__num_leaves': [15, 31, 63],
            'clf__subsample': [0.7, 0.8, 0.9, 1.0],
            'clf__colsample_bytree': [0.6, 0.8, 1.0],
            'clf__reg_alpha': [0, 0.1, 1.0],
            'clf__reg_lambda': [1.0, 2.0, 5.0],
        },
        'n_iter': 40,
    },
    'SVM (RBF)': {
        'pipeline': Pipeline([
            ('scaler', StandardScaler()),
            ('clf', SVC(class_weight='balanced', probability=True, random_state=SEED))
        ]),
        'params': {
            'clf__C': [0.1, 0.5, 1.0, 5.0, 10.0, 50.0],
            'clf__gamma': ['scale', 'auto', 0.01, 0.1, 1.0],
        },
        'n_iter': 20,
    },
    'MLP': {
        'pipeline': Pipeline([
            ('scaler', StandardScaler()),
            ('clf', MLPClassifier(
                early_stopping=True, validation_fraction=0.15,
                random_state=SEED, max_iter=1000))
        ]),
        'params': {
            'clf__hidden_layer_sizes': [(64,), (128,), (64, 32), (128, 64), (64, 32, 16)],
            'clf__alpha': [0.001, 0.01, 0.1],
            'clf__learning_rate_init': [0.001, 0.005, 0.01],
        },
        'n_iter': 20,
    },
}

print(f'Models defined: {list(model_configs.keys())}')
print(f'Outer CV: 5-fold | Inner CV: 3-fold | Seed: {SEED}')

## 3. Nested CV: Tune + Evaluate All Structured-Feature Models

For each outer fold: tune on inner folds, evaluate best config on held-out fold.

In [ ]:
def nested_cv_evaluate(name, config, X, y, outer_cv, inner_cv):
    """Run nested CV: inner for tuning, outer for unbiased evaluation."""
    fold_metrics = []
    best_params_per_fold = []

    # Check if X is a list (text data) vs numpy array (structured data)
    is_text = isinstance(X, list)

    for fold_idx, (train_idx, test_idx) in enumerate(outer_cv.split(np.zeros(len(y)), y)):
        if is_text:
            X_train = [X[i] for i in train_idx]
            X_test = [X[i] for i in test_idx]
        else:
            X_train, X_test = X[train_idx], X[test_idx]
        y_train, y_test = y[train_idx], y[test_idx]

        # Inner CV: hyperparameter tuning
        search = RandomizedSearchCV(
            config['pipeline'], config['params'],
            n_iter=config['n_iter'], cv=inner_cv,
            scoring='average_precision', random_state=SEED,
            n_jobs=1, refit=True
        )
        search.fit(X_train, y_train)
        best_params_per_fold.append(search.best_params_)

        # Evaluate on outer fold
        y_prob = search.predict_proba(X_test)[:, 1]
        y_pred = (y_prob >= 0.5).astype(int)

        fold_metrics.append({
            'roc_auc': roc_auc_score(y_test, y_prob),
            'pr_auc': average_precision_score(y_test, y_prob),
            'f1': f1_score(y_test, y_pred),
            'mcc': matthews_corrcoef(y_test, y_pred),
        })

    metrics_df = pd.DataFrame(fold_metrics)
    return metrics_df, best_params_per_fold

# Run nested CV for all structured-feature models
all_results = {}
for name, config in model_configs.items():
    print(f'Tuning + evaluating: {name}...')
    metrics_df, best_params = nested_cv_evaluate(name, config, X_struct, y, outer_cv, inner_cv)
    all_results[name] = {
        'metrics': metrics_df,
        'best_params': best_params,
    }
    print(f'  PR AUC: {metrics_df["pr_auc"].mean():.3f} +/- {metrics_df["pr_auc"].std():.3f}')
    print(f'  ROC AUC: {metrics_df["roc_auc"].mean():.3f} +/- {metrics_df["roc_auc"].std():.3f}')
    print()

print('All structured-feature models evaluated.')

## 4. Text-Based Classifier (TF-IDF + Logistic Regression)

A text-only classifier using TF-IDF on raw agent responses. This tests whether
linguistic patterns in the raw text carry safety signal beyond the engineered features.

In [ ]:
# TF-IDF text classifier with tuning
text_config = {
    'pipeline': Pipeline([
        ('tfidf', TfidfVectorizer(max_features=5000, ngram_range=(1, 2), sublinear_tf=True)),
        ('clf', LogisticRegression(class_weight='balanced', max_iter=2000, random_state=SEED))
    ]),
    'params': {
        'tfidf__max_features': [2000, 5000, 10000],
        'tfidf__ngram_range': [(1, 1), (1, 2), (1, 3)],
        'tfidf__min_df': [2, 3, 5],
        'clf__C': [0.1, 0.5, 1.0, 5.0, 10.0],
        'clf__penalty': ['l1', 'l2'],
        'clf__solver': ['saga'],
    },
    'n_iter': 30,
}

print('Tuning + evaluating: TF-IDF Text Classifier...')
metrics_df, best_params = nested_cv_evaluate(
    'TF-IDF Text', text_config, X_text, y, outer_cv, inner_cv
)
all_results['TF-IDF Text Classifier'] = {
    'metrics': metrics_df,
    'best_params': best_params,
}
print(f'  PR AUC: {metrics_df["pr_auc"].mean():.3f} +/- {metrics_df["pr_auc"].std():.3f}')
print(f'  ROC AUC: {metrics_df["roc_auc"].mean():.3f} +/- {metrics_df["roc_auc"].std():.3f}')
print(f'\nText classifier evaluated.')

## 5. Summary Table - All Models

In [ ]:
# Build comparison table
summary_rows = []
for name, result in all_results.items():
    m = result['metrics']
    summary_rows.append({
        'Model': name,
        'ROC AUC': f'{m["roc_auc"].mean():.3f} +/- {m["roc_auc"].std():.3f}',
        'PR AUC': f'{m["pr_auc"].mean():.3f} +/- {m["pr_auc"].std():.3f}',
        'F1': f'{m["f1"].mean():.3f} +/- {m["f1"].std():.3f}',
        'MCC': f'{m["mcc"].mean():.3f} +/- {m["mcc"].std():.3f}',
        'roc_auc_mean': m['roc_auc'].mean(),
        'pr_auc_mean': m['pr_auc'].mean(),
        'f1_mean': m['f1'].mean(),
        'mcc_mean': m['mcc'].mean(),
    })

summary_df = pd.DataFrame(summary_rows).set_index('Model')
summary_df = summary_df.sort_values('pr_auc_mean', ascending=False)

print('=== Advanced Model Comparison (Nested CV, Tuned) ===\n')
print(summary_df[['ROC AUC', 'PR AUC', 'F1', 'MCC']].to_string())

best_model = summary_df['pr_auc_mean'].idxmax()
print(f'\nBest model by PR AUC: {best_model} ({summary_df.loc[best_model, "pr_auc_mean"]:.3f})')

## 6. Visualization - Model Comparison

In [ ]:
# Dot plot with error bars for all models and metrics
fig, axes = plt.subplots(1, 4, figsize=(18, 5), sharey=True)
metrics_plot = ['roc_auc', 'pr_auc', 'f1', 'mcc']
metric_titles = ['ROC AUC', 'PR AUC', 'F1', 'MCC']

# Sort models by PR AUC for consistent ordering
model_order = summary_df.index.tolist()

for i, (metric, title) in enumerate(zip(metrics_plot, metric_titles)):
    means = [all_results[m]['metrics'][metric].mean() for m in model_order]
    stds = [all_results[m]['metrics'][metric].std() for m in model_order]

    y_pos = range(len(model_order))
    axes[i].errorbar(means, y_pos, xerr=stds, fmt='o', color='#2980b9',
                     capsize=4, markersize=8, linewidth=2)
    axes[i].set_yticks(y_pos)
    if i == 0:
        axes[i].set_yticklabels(model_order, fontsize=9)
    axes[i].set_title(title, fontsize=11)
    axes[i].axvline(x=0.5, ls='--', color='grey', alpha=0.3)
    axes[i].set_xlim(0, 1)

plt.suptitle('Advanced Model Comparison (Nested CV, Tuned)', y=1.02, fontsize=13)
plt.tight_layout()
plt.savefig(FIGURES_DIR / '14_advanced_model_comparison.png', bbox_inches='tight')
plt.show()

## 7. Log Results to JSON

In [ ]:
# Structured logging of all results
log_entry = {
    'step': 17,
    'description': 'Advanced model comparison with nested CV and hyperparameter tuning',
    'timestamp': datetime.now().isoformat(),
    'seed': SEED,
    'n_samples': len(y),
    'n_unsafe': int(y.sum()),
    'n_safe': int((y == 0).sum()),
    'outer_cv_folds': 5,
    'inner_cv_folds': 3,
    'feature_type': {
        'structured': {'n_features': len(feature_cols), 'features': feature_cols},
        'text': {'method': 'TF-IDF', 'source': 'concatenated assistant turns'},
    },
    'models': {},
}

for name, result in all_results.items():
    m = result['metrics']
    # Convert best_params to serializable format
    params_serializable = []
    for p in result['best_params']:
        params_serializable.append({k: str(v) for k, v in p.items()})

    log_entry['models'][name] = {
        'metrics': {
            'roc_auc': {'mean': float(m['roc_auc'].mean()), 'std': float(m['roc_auc'].std())},
            'pr_auc': {'mean': float(m['pr_auc'].mean()), 'std': float(m['pr_auc'].std())},
            'f1': {'mean': float(m['f1'].mean()), 'std': float(m['f1'].std())},
            'mcc': {'mean': float(m['mcc'].mean()), 'std': float(m['mcc'].std())},
        },
        'best_params_per_fold': params_serializable,
    }

log_entry['best_model'] = best_model
log_entry['best_pr_auc'] = float(summary_df.loc[best_model, 'pr_auc_mean'])

# Save
results_path = RESULTS_DIR / 'model_comparison.json'
with open(results_path, 'w') as f:
    json.dump(log_entry, f, indent=2)

print(f'Results logged to: {results_path}')
print(f'Best model: {best_model} (PR AUC = {log_entry["best_pr_auc"]:.3f})')

## 8. Summary

**Key takeaways:**
- Nested CV provides unbiased performance estimates (no data leakage from tuning)
- XGBoost/LightGBM leverage dedicated gradient boosting implementations
- Text-based classifier tests whether raw linguistic patterns carry additional safety signal
- All models tuned via RandomizedSearchCV with class-imbalance handling
- Results logged for reproducibility and downstream reporting